# OncoEvidence — Self-Contained End-to-End Notebook (Google Colab)

**Mechanism-Guided AI Evidence Triage for Cancer Drug Repurposing**

This notebook is **fully self-contained**: it does **not** clone or import the
project source package. Every component — PrimeKG parsing, node features, the
HeteroGNN / XGBoost / MLP / KGE models, the leakage-safe splits, the trainer,
the multi-hop mechanism-path extractor, the DrugMechDB/UniProt mapping, the
Europe PMC retrieval, and the LLM/lexical verifier — is **defined inline below**.

The only external dependencies are:
- pip packages (PyTorch, PyG, XGBoost, Optuna, SentenceTransformers, …), and
- public **data/APIs**: the PrimeKG download (Harvard Dataverse), Europe PMC,
  DrugMechDB, and mygene.info. (These are data sources, not source code.)

**Workflow:** setup → PrimeKG → Aim 1 (link benchmark + ablations) → Aim 2
(mechanism paths) → Aim 4 (true-vs-random separation + hard negatives) → Aim 3
(LLM/lexical verification) → Finding 4 (blinded mechanism recovery) → deliverable
shortlist.

Switches in section 0: `FAST_MODE` (quick pass vs full published reproduction) and
`RUN_LLM` (enable the LLM reviewer).

> Recommended: **Runtime → Change runtime type → GPU**.
> *All predictions are hypothesis-generating and not medical advice.*

## 0. Configuration switches

In [ ]:
FAST_MODE = True   # True: quick pass (1 seed, reduced epochs, hashing features). False: full reproduction (hours).
RUN_LLM   = False  # True: use the LLM reviewer/judge (needs ONCO_LLM_API_KEY below).

import os
os.environ.setdefault("ONCO_LLM_API_KEY", "")               # paste an OpenAI-compatible key to enable LLM
ONCO_LLM_BASE_URL = "https://openrouter.ai/api/v1"          # OpenAI default: https://api.openai.com/v1
ONCO_LLM_MODEL    = "openai/gpt-4o-mini"
if RUN_LLM and os.environ["ONCO_LLM_API_KEY"]:
    os.environ["ONCO_LLM_BASE_URL"] = ONCO_LLM_BASE_URL
    os.environ["ONCO_LLM_MODEL"]    = ONCO_LLM_MODEL

# Training/eval budgets.
SEEDS             = [0] if FAST_MODE else [0, 1, 2, 42, 123]
ABLATION_SEEDS    = [0] if FAST_MODE else [0, 1, 2]
GNN_EPOCHS        = 12 if FAST_MODE else 50
MLP_EPOCHS        = 50 if FAST_MODE else 200
KGE_EPOCHS        = 80 if FAST_MODE else 300
PATIENCE          = 6 if FAST_MODE else 10
XGB_TUNE          = (not FAST_MODE)
XGB_TRIALS        = 5 if FAST_MODE else 8
XGB_ESTIMATORS    = 200 if FAST_MODE else 400
USE_FALLBACK_FEATS = FAST_MODE       # hashing features in fast mode (instant); ST embeddings in full mode
N_SEP             = 120 if FAST_MODE else 400
REC_EPOCHS        = 12 if FAST_MODE else 60
REC_SEEDS         = [0] if FAST_MODE else [0, 1, 2]
print(f"FAST_MODE={FAST_MODE} RUN_LLM={RUN_LLM} | seeds={SEEDS} gnn_epochs={GNN_EPOCHS} xgb_tune={XGB_TUNE}")

## 1. Install dependencies

On Colab PyTorch is preinstalled; we add the extras only.

In [ ]:
%pip install -q torch-geometric xgboost optuna "sentence-transformers>=2.2.0" "transformers>=4.40,<5" scikit-learn pandas numpy scipy matplotlib seaborn requests tqdm pyyaml
print("Dependencies installed.")

In [ ]:
import torch
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Inline training is slow on CPU; use Runtime > Change runtime type > GPU.")

---
# Inlined library

The cells below define the **entire pipeline inline**. Run them once (top to
bottom); later workflow sections call into them. There are no `import oncorepurpose`
statements anywhere.

### L1 · Paths, schema constants, oncology keywords

In [ ]:
from pathlib import Path

DATA_DIR    = Path("data");    MODELS_DIR = Path("models")
RESULTS_DIR = Path("results"); FIGURES_DIR = Path("figures")
for _d in (DATA_DIR, MODELS_DIR, RESULTS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PRIMEKG_KG_CSV = DATA_DIR / "kg.csv"
PRIMEKG_KG_URL = "https://dataverse.harvard.edu/api/access/datafile/6180620"
HETERODATA_PT  = DATA_DIR / "primekg_hetero.pt"
FEATURE_CACHE  = MODELS_DIR / "primekg_text_features.pt"
LLM_CACHE_DIR  = MODELS_DIR / "llm_cache"
UNIPROT_CACHE  = DATA_DIR / "uniprot2symbol.json"

INDICATION_REL = "indication"; CONTRAINDICATION_REL = "contraindication"; OFFLABEL_REL = "off-label use"
THERAPEUTIC_RELS = (INDICATION_REL, CONTRAINDICATION_REL, OFFLABEL_REL)
DRUG_TYPE = "drug"; DISEASE_TYPE = "disease"; GENE_TYPE = "gene_protein"
TEXT_MODEL = "all-MiniLM-L6-v2"
ONCOLOGY_KEYWORDS = (
    "cancer","carcinoma","sarcoma","neoplasm","tumor","tumour","leukemia","leukaemia",
    "lymphoma","melanoma","glioma","glioblastoma","myeloma","blastoma","adenoma",
    "malignant","metastat","oncocytoma","mesothelioma","astrocytoma","teratoma","carcinosarcoma",
)
print("constants ready")

### L2 · PrimeKG download + HeteroData build

In [ ]:
import sys, re, urllib.request
from collections import defaultdict
import pandas as pd
import torch
from torch_geometric.data import HeteroData

def download_primekg(dest=PRIMEKG_KG_CSV, force=False):
    dest = Path(dest)
    if dest.exists() and not force and dest.stat().st_size > 0:
        print(f"PrimeKG present: {dest} ({dest.stat().st_size/1e6:.0f} MB)"); return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    print(f"Downloading PrimeKG -> {dest} (~980 MB)")
    def _p(b, bs, t):
        if t > 0:
            sys.stdout.write(f"\r  {min(100.0, b*bs*100.0/t):5.1f}%"); sys.stdout.flush()
    urllib.request.urlretrieve(PRIMEKG_KG_URL, dest, _p); sys.stdout.write("\n")
    print(f"Saved {dest} ({dest.stat().st_size/1e6:.0f} MB)"); return dest

def _norm_type(t): return re.sub(r"[^0-9a-zA-Z]+", "_", str(t).strip().lower()).strip("_")
def _norm_rel(r):  return re.sub(r"[^0-9a-zA-Z]+", "_", str(r).strip().lower()).strip("_")
def _is_oncology(name): return any(k in str(name).lower() for k in ONCOLOGY_KEYWORDS)

def build_hetero_from_primekg(kg_csv=PRIMEKG_KG_CSV, output_path=HETERODATA_PT, save=True):
    kg_csv = Path(kg_csv)
    if not kg_csv.exists():
        raise FileNotFoundError(f"{kg_csv} not found; run download_primekg() first.")
    print(f"Loading {kg_csv} ...")
    df = pd.read_csv(kg_csv, dtype=str, low_memory=False)
    print(f"  {len(df):,} raw edges")
    df["x_type_n"] = df["x_type"].map(_norm_type); df["y_type_n"] = df["y_type"].map(_norm_type)
    df["rel_n"] = df["relation"].map(_norm_rel)
    node_id_to_idx = defaultdict(dict); node_names = defaultdict(list); node_ids = defaultdict(list)
    def _register(t, i, nm):
        m = node_id_to_idx[t]
        if i not in m:
            m[i] = len(m); node_names[t].append("" if nm is None else str(nm)); node_ids[t].append(str(i))
        return m[i]
    for side in ("x", "y"):
        sub = df[[f"{side}_type_n", f"{side}_id", f"{side}_name"]].drop_duplicates()
        for t, i, nm in sub.itertuples(index=False):
            _register(t, i, nm)
    data = HeteroData()
    for nt, m in node_id_to_idx.items():
        data[nt].num_nodes = len(m); data[nt].node_ids = node_ids[nt]; data[nt].node_names = node_names[nt]
    buckets = defaultdict(list)
    for x_t, x_id, y_t, y_id, rel in df[["x_type_n","x_id","y_type_n","y_id","rel_n"]].itertuples(index=False):
        buckets[(x_t, rel, y_t)].append((node_id_to_idx[x_t][x_id], node_id_to_idx[y_t][y_id]))
    for et, pairs in buckets.items():
        data[et].edge_index = torch.tensor(pairs, dtype=torch.long).t().contiguous()
    if DISEASE_TYPE in data.node_types:
        names = data[DISEASE_TYPE].node_names
        data[DISEASE_TYPE].is_oncology = torch.tensor([_is_oncology(n) for n in names], dtype=torch.bool)
        print(f"  oncology diseases: {int(data[DISEASE_TYPE].is_oncology.sum())} / {len(names)}")
    tn = sum(int(data[t].num_nodes) for t in data.node_types)
    te = sum(int(data[e].edge_index.size(1)) for e in data.edge_types)
    print(f"  {len(data.node_types)} node types ({tn:,} nodes), {len(data.edge_types)} edge types ({te:,} edges)")
    if save:
        torch.save(data, Path(output_path)); print(f"Saved -> {output_path}")
    return data
print("L2 ready")

### L3 · Shared node features (SentenceTransformer + hashing fallback)

In [ ]:
import hashlib
import numpy as np

DEFAULT_HASH_DIM = 256

def _node_texts(data, nt):
    store = data[nt]; names = getattr(store, "node_names", None); n = int(store.num_nodes)
    pretty = nt.replace("_", " ")
    if names is None: return [f"{pretty} {i}" for i in range(n)]
    out = []
    for i in range(n):
        raw = "" if (i >= len(names) or names[i] is None) else str(names[i]).strip()
        out.append(f"{pretty}: {raw}" if raw else f"{pretty} {i}")
    return out

def _hash_embed(texts, dim=DEFAULT_HASH_DIM):
    vecs = np.zeros((len(texts), dim), dtype=np.float32)
    for row, text in enumerate(texts):
        toks = re.findall(r"[a-z0-9]+", text.lower()); grams = list(toks)
        for tok in toks:
            p = f"#{tok}#"; grams.extend(p[i:i+3] for i in range(len(p)-2))
        for g in grams:
            h = int(hashlib.md5(g.encode()).hexdigest(), 16)
            vecs[row, h % dim] += 1.0 if (h//dim) % 2 == 0 else -1.0
    norms = np.linalg.norm(vecs, axis=1, keepdims=True); norms[norms == 0] = 1.0
    return vecs / norms

def _st_embed(texts_by_type, model_name):
    try:
        from sentence_transformers import SentenceTransformer
    except Exception as exc:
        print(f"  [features] sentence-transformers unavailable ({exc}); hashing fallback"); return None
    try:
        model = SentenceTransformer(model_name); out = {}
        for nt, texts in texts_by_type.items():
            out[nt] = model.encode(texts, batch_size=512, normalize_embeddings=True,
                                   show_progress_bar=False, convert_to_numpy=True).astype(np.float32)
        return out
    except Exception as exc:
        print(f"  [features] SentenceTransformer failed ({exc}); hashing fallback"); return None

def build_text_features(data, cache_path=FEATURE_CACHE, model_name=TEXT_MODEL, force_fallback=False):
    cache_path = Path(cache_path) if cache_path is not None else None
    if cache_path is not None and force_fallback:
        cache_path = cache_path.with_name(cache_path.stem + "_hash" + cache_path.suffix)
    if cache_path is not None and cache_path.exists():
        cached = torch.load(cache_path, weights_only=False)
        if all(nt in cached and cached[nt].shape[0] == int(data[nt].num_nodes) for nt in data.node_types):
            for nt in data.node_types: data[nt].x = cached[nt].float()
            print(f"  [features] loaded cache (dim={next(iter(cached.values())).shape[1]})"); return data
    texts = {nt: _node_texts(data, nt) for nt in data.node_types}
    emb = None if force_fallback else _st_embed(texts, model_name)
    if emb is None:
        emb = {nt: _hash_embed(t) for nt, t in texts.items()}; src = f"hashing (dim={DEFAULT_HASH_DIM})"
    else:
        src = f"{model_name} (dim={next(iter(emb.values())).shape[1]})"
    tensors = {}
    for nt in data.node_types:
        x = torch.from_numpy(np.ascontiguousarray(emb[nt])).float(); data[nt].x = x; tensors[nt] = x
    print(f"  [features] built via {src}")
    if cache_path is not None:
        cache_path.parent.mkdir(parents=True, exist_ok=True); torch.save(tensors, cache_path)
    return data

def _find_target_edge(data, relation):
    rel_n = _norm_rel(relation); cands = []
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r == rel_n: cands.append(et)
    if not cands: return None
    for et in cands:
        if et[0] == DRUG_TYPE: return et
    return cands[0]

def load_primekg(pt_path=HETERODATA_PT, with_features=True, force_fallback_features=False, build_if_missing=True):
    pt_path = Path(pt_path)
    if not pt_path.exists():
        if not build_if_missing: raise FileNotFoundError(f"{pt_path} not found.")
        data = build_hetero_from_primekg(save=True)
    else:
        import warnings
        with warnings.catch_warnings():
            warnings.simplefilter("ignore"); data = torch.load(pt_path, weights_only=False)
    for nt in data.node_types:
        if not hasattr(data[nt], "num_nodes") or data[nt].num_nodes is None:
            names = getattr(data[nt], "node_names", None)
            data[nt].num_nodes = len(names) if names is not None else 0
    if with_features:
        build_text_features(data, force_fallback=force_fallback_features)
    targets = {"indication": _find_target_edge(data, INDICATION_REL),
               "contraindication": _find_target_edge(data, CONTRAINDICATION_REL)}
    return data, targets

def graph_summary(data, targets):
    lines = ["PrimeKG graph:"]; tn = te = 0
    for nt in data.node_types:
        n = int(data[nt].num_nodes); tn += n
        dim = int(data[nt].x.size(1)) if "x" in data[nt] else None
        extra = f" feat={dim}" if dim else ""
        if nt == DISEASE_TYPE and "is_oncology" in data[nt]:
            extra += f" oncology={int(data[nt].is_oncology.sum())}"
        lines.append(f"  node {nt}: {n}{extra}")
    for et in data.edge_types: te += int(data[et].edge_index.size(1))
    lines.append(f"  totals: {tn:,} nodes, {te:,} edges, {len(data.edge_types)} edge types")
    for name, et in targets.items():
        lines.append(f"  target [{name}]: {et} = {int(data[et].edge_index.size(1))} edges" if et else f"  target [{name}]: NOT FOUND")
    return "\n".join(lines)
print("L3 ready")

### L4 · Models (HeteroGNN, FeatureMLP, DistMultKGE)

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import HeteroConv, SAGEConv

class EdgeMLPDecoder(nn.Module):
    def __init__(self, hidden):
        super().__init__(); self.net = nn.Sequential(nn.Linear(2*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, zs, zd): return self.net(torch.cat([zs, zd], dim=-1)).squeeze(-1)

class MechanismHead(nn.Module):
    def __init__(self, hidden):
        super().__init__(); self.net = nn.Sequential(nn.Linear(3*hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, zd, zg, zc): return self.net(torch.cat([zd, zg, zc], dim=-1)).squeeze(-1)

class HeteroGNN(nn.Module):
    def __init__(self, node_types, edge_types, in_dims, hidden=128, num_layers=2, dropout=0.3):
        super().__init__()
        self.node_types = node_types; self.edge_types = edge_types; self.dropout = dropout
        self.proj = nn.ModuleDict({nt: nn.Linear(in_dims[nt], hidden) for nt in node_types})
        self.convs = nn.ModuleList([HeteroConv({et: SAGEConv(hidden, hidden) for et in edge_types}, aggr="sum")
                                    for _ in range(num_layers)])
        self.decoder = EdgeMLPDecoder(hidden); self.mech_head = MechanismHead(hidden)
    def encode(self, data):
        device = next(self.parameters()).device
        x_dict = {nt: self.proj[nt](data[nt].x.to(device)) for nt in self.node_types if nt in data.node_types}
        eid = {et: data[et].edge_index.to(device) for et in self.edge_types
               if et in data.edge_types and data[et].edge_index.numel() > 0}
        for conv in self.convs:
            out = conv(x_dict, eid)
            for nt in x_dict:
                if nt not in out: out[nt] = x_dict[nt]
            x_dict = {nt: F.dropout(F.relu(v), p=self.dropout, training=self.training) for nt, v in out.items()}
        return x_dict
    def decode(self, z, et, eli):
        s_t, _, d_t = et; dev = z[s_t].device; eli = eli.to(dev)
        return self.decoder(z[s_t][eli[0]], z[d_t][eli[1]])
    def score_mechanism(self, z, di, gi, ci, dt="drug", gt="gene_protein", ct="disease"):
        dev = z[dt].device
        return self.mech_head(z[dt][di.to(dev)], z[gt][gi.to(dev)], z[ct][ci.to(dev)])

class FeatureMLP(nn.Module):
    def __init__(self, node_types, in_dims, hidden=128, dropout=0.3):
        super().__init__(); self.node_types = node_types; self.dropout = dropout
        self.proj = nn.ModuleDict({nt: nn.Linear(in_dims[nt], hidden) for nt in node_types})
        self.decoder = EdgeMLPDecoder(hidden)
    def encode(self, data):
        device = next(self.parameters()).device
        return {nt: F.relu(self.proj[nt](data[nt].x.to(device))) for nt in self.node_types if nt in data.node_types}
    def decode(self, z, et, eli):
        s_t, _, d_t = et; dev = z[s_t].device; eli = eli.to(dev)
        return self.decoder(z[s_t][eli[0]], z[d_t][eli[1]])

class DistMultKGE(nn.Module):
    def __init__(self, src_type, dst_type, num_src, num_dst, dim=128):
        super().__init__(); self.src_type, self.dst_type = src_type, dst_type
        self.src_emb = nn.Embedding(num_src, dim); self.dst_emb = nn.Embedding(num_dst, dim)
        self.rel = nn.Parameter(torch.randn(dim)*0.1)
        nn.init.xavier_uniform_(self.src_emb.weight); nn.init.xavier_uniform_(self.dst_emb.weight)
    def score(self, eli):
        dev = self.rel.device; eli = eli.to(dev)
        return (self.src_emb(eli[0]) * self.rel * self.dst_emb(eli[1])).sum(-1)
print("L4 ready")

### L5 · Metrics

In [ ]:
from sklearn.metrics import (average_precision_score, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score)

def _to_np(x):
    if isinstance(x, torch.Tensor): return x.detach().cpu().numpy()
    if isinstance(x, list): return np.array(x)
    return x

def hits_at_k(y, s, k=10):
    y, s = _to_np(y), _to_np(s); order = np.argsort(-s)[:k]; tp = float(np.sum(y))
    return 0.0 if tp == 0 else float(np.sum(y[order]) / min(k, tp))

def mean_reciprocal_rank(y, s):
    y, s = _to_np(y), _to_np(s); ranked = y[np.argsort(-s)]; pr = np.where(ranked == 1)[0] + 1
    return 0.0 if pr.size == 0 else float(np.mean(1.0/pr))

def optimal_threshold_metrics(y, s):
    prec, rec, thr = precision_recall_curve(y, s)
    f1 = 2*(prec[:-1]*rec[:-1])/(prec[:-1]+rec[:-1]+1e-10)
    if f1.size == 0: return {"precision":0.0,"recall":0.0,"f1":0.0,"threshold":0.5}
    i = int(np.argmax(f1)); t = float(thr[i]); pred = (s >= t).astype(int)
    return {"precision":float(precision_score(y,pred,zero_division=0)),
            "recall":float(recall_score(y,pred,zero_division=0)),
            "f1":float(f1_score(y,pred,zero_division=0)),"threshold":t}

def compute_all_metrics(y, s, prefix=""):
    y, s = _to_np(y), _to_np(s)
    keys = ["auroc","auprc","hits@1","hits@3","hits@10","mrr","precision","recall","f1"]
    if len(y)==0 or np.sum(y)==0 or np.sum(y)==len(y): return {f"{prefix}{k}":0.0 for k in keys}
    tm = optimal_threshold_metrics(y, s)
    return {f"{prefix}auroc":float(roc_auc_score(y,s)), f"{prefix}auprc":float(average_precision_score(y,s)),
            f"{prefix}hits@1":hits_at_k(y,s,1), f"{prefix}hits@3":hits_at_k(y,s,3),
            f"{prefix}hits@10":hits_at_k(y,s,10), f"{prefix}mrr":mean_reciprocal_rank(y,s),
            f"{prefix}precision":tm["precision"], f"{prefix}recall":tm["recall"], f"{prefix}f1":tm["f1"]}
print("L5 ready")

### L6 · Leakage-safe splits (transductive / inductive) + ablations

In [ ]:
from dataclasses import dataclass, field
from typing import Set, Tuple, List, Optional
_THERAPEUTIC_NORM = {_norm_rel(r) for r in THERAPEUTIC_RELS}

@dataclass
class SplitData:
    base: HeteroData; target_edge_type: tuple; regime: str
    train_label_index: torch.Tensor; train_label: torch.Tensor
    val_label_index: torch.Tensor; val_label: torch.Tensor
    test_label_index: torch.Tensor; test_label: torch.Tensor
    info: dict = field(default_factory=dict)

def _therapeutic_edge_types(data):
    out = []
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r in _THERAPEUTIC_NORM: out.append(et)
    return out

def _sample_negatives(pos_set, src_pool, dst_pool, num, gen):
    if num <= 0 or src_pool.numel() == 0 or dst_pool.numel() == 0:
        return torch.empty((2,0), dtype=torch.long)
    ns, nd, seen = [], [], set(); attempts = 0; maxa = max(2000, num*50)
    while len(ns) < num and attempts < maxa:
        batch = max(num*2, 128)
        si = src_pool[torch.randint(0, src_pool.numel(), (batch,), generator=gen)]
        di = dst_pool[torch.randint(0, dst_pool.numel(), (batch,), generator=gen)]
        for s, d in zip(si.tolist(), di.tolist()):
            key = (s, d)
            if key in pos_set or key in seen: continue
            seen.add(key); ns.append(s); nd.append(d)
            if len(ns) >= num: break
        attempts += batch
    return torch.tensor([ns, nd], dtype=torch.long)

def _build_base_graph(data, target_edge_type, train_target_cols):
    base = HeteroData()
    for nt in data.node_types:
        for k, v in data[nt].items(): base[nt][k] = v
    therapeutic = set(_therapeutic_edge_types(data))
    for et in data.edge_types:
        if et == target_edge_type: base[et].edge_index = data[et].edge_index[:, train_target_cols]
        elif et in therapeutic:    base[et].edge_index = data[et].edge_index[:, :0]
        else:                       base[et].edge_index = data[et].edge_index
    return base

def _make(base, target, regime, tr_pos, tr_neg, va_pos, va_neg, te_pos, te_neg, info):
    def cat(p, n):
        return torch.cat([p, n], dim=1), torch.cat([torch.ones(p.size(1)), torch.zeros(n.size(1))])
    tr_i, tr_l = cat(tr_pos, tr_neg); va_i, va_l = cat(va_pos, va_neg); te_i, te_l = cat(te_pos, te_neg)
    return SplitData(base, target, regime, tr_i, tr_l, va_i, va_l, te_i, te_l, info)

def transductive_split(data, target, val_frac=0.1, test_frac=0.2, neg_ratio=1.0, seed=0):
    gen = torch.Generator().manual_seed(seed); s_t, _, d_t = target
    pos = data[target].edge_index; n = pos.size(1); perm = torch.randperm(n, generator=gen)
    n_test, n_val = int(round(test_frac*n)), int(round(val_frac*n))
    test_c, val_c, train_c = perm[:n_test], perm[n_test:n_test+n_val], perm[n_test+n_val:]
    tr_pos, va_pos, te_pos = pos[:, train_c], pos[:, val_c], pos[:, test_c]
    pos_set = {(int(s), int(d)) for s, d in zip(pos[0].tolist(), pos[1].tolist())}
    src_pool = torch.arange(int(data[s_t].num_nodes)); dst_pool = torch.arange(int(data[d_t].num_nodes))
    base = _build_base_graph(data, target, train_c)
    def neg(p): return _sample_negatives(pos_set, src_pool, dst_pool, int(round(p.size(1)*neg_ratio)), gen)
    info = {"regime":"transductive","train_pos":int(tr_pos.size(1)),"val_pos":int(va_pos.size(1)),"test_pos":int(te_pos.size(1))}
    return _make(base, target, "transductive", tr_pos, neg(tr_pos), va_pos, neg(va_pos), te_pos, neg(te_pos), info)

def inductive_node_split(data, target, holdout_side="dst", val_frac=0.1, test_frac=0.2,
                         neg_ratio=1.0, seed=0, restrict_oncology=False):
    gen = torch.Generator().manual_seed(seed); s_t, _, d_t = target
    pos = data[target].edge_index; side_row = 0 if holdout_side == "src" else 1
    side_type = s_t if holdout_side == "src" else d_t
    participating = torch.unique(pos[side_row])
    if restrict_oncology and holdout_side == "dst" and "is_oncology" in data[d_t]:
        onc = data[d_t].is_oncology
        participating = torch.tensor([n for n in participating.tolist() if bool(onc[n])], dtype=torch.long)
    perm = participating[torch.randperm(participating.numel(), generator=gen)]; n = participating.numel()
    n_test, n_val = max(1,int(round(test_frac*n))), max(1,int(round(val_frac*n)))
    test_nodes = set(perm[:n_test].tolist()); val_nodes = set(perm[n_test:n_test+n_val].tolist()); held = test_nodes|val_nodes
    side_idx = pos[side_row].tolist(); tr_c, va_c, te_c = [], [], []
    for col, nd in enumerate(side_idx):
        (te_c if nd in test_nodes else va_c if nd in val_nodes else tr_c).append(col)
    tr_c = torch.tensor(tr_c, dtype=torch.long); tr_pos = pos[:, tr_c]
    va_pos = pos[:, torch.tensor(va_c, dtype=torch.long)] if va_c else pos[:, :0]
    te_pos = pos[:, torch.tensor(te_c, dtype=torch.long)] if te_c else pos[:, :0]
    pos_set = {(int(s), int(d)) for s, d in zip(pos[0].tolist(), pos[1].tolist())}
    other_type = d_t if holdout_side == "src" else s_t; other_pool = torch.arange(int(data[other_type].num_nodes))
    train_side = torch.tensor(sorted(set(participating.tolist())-held), dtype=torch.long)
    base = _build_base_graph(data, target, tr_c)
    def neg(p, side_pool):
        num = int(round(p.size(1)*neg_ratio))
        return (_sample_negatives(pos_set, side_pool, other_pool, num, gen) if holdout_side == "src"
                else _sample_negatives(pos_set, other_pool, side_pool, num, gen))
    def pool(s): return torch.tensor(sorted(s), dtype=torch.long)
    info = {"regime":f"inductive_cold_{holdout_side}","restrict_oncology":restrict_oncology,
            "n_test_nodes":len(test_nodes),"n_val_nodes":len(val_nodes),
            "train_pos":int(tr_pos.size(1)),"val_pos":int(va_pos.size(1)),"test_pos":int(te_pos.size(1))}
    return _make(base, target, info["regime"], tr_pos, neg(tr_pos, train_side),
                 va_pos, neg(va_pos, pool(val_nodes)), te_pos, neg(te_pos, pool(test_nodes)), info)

def make_split(data, target, regime, seed=0, holdout_side="dst", val_frac=0.1, test_frac=0.2,
               neg_ratio=1.0, restrict_oncology=False):
    if regime == "transductive": return transductive_split(data, target, val_frac, test_frac, neg_ratio, seed)
    if regime in ("inductive","inductive_cold_dst","inductive_cold_src"):
        side = "src" if regime.endswith("src") else holdout_side
        return inductive_node_split(data, target, side, val_frac, test_frac, neg_ratio, seed, restrict_oncology)
    raise ValueError(regime)

def ablate_topology(split, mode, seed=0):
    gen = torch.Generator().manual_seed(seed); nb = HeteroData()
    for nt in split.base.node_types:
        for k, v in split.base[nt].items(): nb[nt][k] = v
    for et in split.base.edge_types:
        ei = split.base[et].edge_index
        if et == split.target_edge_type: nb[et].edge_index = ei; continue
        if mode == "empty": nb[et].edge_index = ei[:, :0]
        elif mode == "shuffle":
            perm = torch.randperm(ei.size(1), generator=gen); nb[et].edge_index = torch.stack([ei[0], ei[1][perm]], 0)
        else: raise ValueError(mode)
    return SplitData(nb, split.target_edge_type, f"{split.regime}_ablate_{mode}", split.train_label_index,
                     split.train_label, split.val_label_index, split.val_label, split.test_label_index,
                     split.test_label, dict(split.info))

def drop_relations(split, drop_substrings):
    nb = HeteroData()
    for nt in split.base.node_types:
        for k, v in split.base[nt].items(): nb[nt][k] = v
    for et in split.base.edge_types:
        _, rel, _ = et
        if et != split.target_edge_type and any(sub in rel for sub in drop_substrings): continue
        nb[et].edge_index = split.base[et].edge_index
    return SplitData(nb, split.target_edge_type, f"{split.regime}_drop", split.train_label_index,
                     split.train_label, split.val_label_index, split.val_label, split.test_label_index,
                     split.test_label, dict(split.info))
print("L6 ready")

### L7 · Trainer (GNN / MLP / KGE, joint mechanism loss, evaluate)

In [ ]:
import copy, random

def set_all_seeds(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

@torch.no_grad()
def _eval_encoder(model, base, et, eli, labels, device):
    model.eval(); z = model.encode(base); scores = torch.sigmoid(model.decode(z, et, eli)).cpu()
    return compute_all_metrics(labels.cpu(), scores)

def train_gnn(model, split, device, epochs=50, patience=10, lr=5e-3, weight_decay=1e-5):
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        loss = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_encoder(model, split.base, et, split.val_label_index, split.val_label, device)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

def train_mlp(model, split, device, epochs=200, patience=20, lr=5e-3, weight_decay=1e-5):
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        loss = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_encoder(model, split.base, et, split.val_label_index, split.val_label, device)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

@torch.no_grad()
def _eval_kge(model, eli, labels):
    model.eval(); return compute_all_metrics(labels.cpu(), torch.sigmoid(model.score(eli)).cpu())

def train_kge(model, split, device, epochs=300, patience=30, lr=1e-2, weight_decay=1e-6):
    model = model.to(device); opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); best, best_state, wait = -1.0, None, 0
    for _ in range(epochs):
        model.train(); opt.zero_grad()
        loss = F.binary_cross_entropy_with_logits(model.score(split.train_label_index), y)
        loss.backward(); opt.step()
        val = _eval_kge(model, split.val_label_index, split.val_label)["auroc"]
        if val > best: best, wait = val, 0; best_state = copy.deepcopy({k: v.cpu() for k, v in model.state_dict().items()})
        else:
            wait += 1
            if wait >= patience: break
    if best_state is not None: model.load_state_dict(best_state)
    return model

def train_gnn_joint(model, split, device, mech, decoys, lam=1.0, n_neg=8, mech_batch=256, epochs=60, lr=5e-3, weight_decay=1e-5):
    model = model.to(device); et = split.target_edge_type
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    y = split.train_label.float().to(device); md_, mc, mg = mech.drug, mech.dis, mech.gene; M = int(md_.numel())
    for _ in range(epochs):
        model.train(); opt.zero_grad(); z = model.encode(split.base)
        link = F.binary_cross_entropy_with_logits(model.decode(z, et, split.train_label_index), y)
        mech_loss = torch.tensor(0.0, device=device)
        if M > 0 and lam > 0:
            b = min(mech_batch, M); idx = torch.randint(0, M, (b,))
            bd, bc, bg = md_[idx], mc[idx], mg[idx]
            neg = torch.tensor([decoys.sample(int(bg[j]), {int(bg[j])}, n_neg) for j in range(b)], dtype=torch.long)
            cand = torch.cat([bg.view(-1,1), neg], dim=1); K = cand.size(1)
            fd = bd.view(-1,1).expand(-1,K).reshape(-1); fc = bc.view(-1,1).expand(-1,K).reshape(-1); fg = cand.reshape(-1)
            logits = model.score_mechanism(z, fd, fg, fc).view(b, K)
            mech_loss = F.cross_entropy(logits, torch.zeros(b, dtype=torch.long, device=logits.device))
        (link + lam*mech_loss).backward(); opt.step()
    return model

def evaluate_model(model, split, device, which="test"):
    eli = getattr(split, f"{which}_label_index"); lab = getattr(split, f"{which}_label")
    if isinstance(model, DistMultKGE): return _eval_kge(model, eli, lab)
    return _eval_encoder(model, split.base, split.target_edge_type, eli, lab, device)
print("L7 ready")

### L8 · Tuned XGBoost tabular baseline

In [ ]:
def _pair_features(data, et, eli):
    s_t, _, d_t = et
    return np.concatenate([data[s_t].x[eli[0]].cpu().numpy(), data[d_t].x[eli[1]].cpu().numpy()], axis=1)

def run_xgboost(split, data, seed=0, n_estimators=400, max_depth=6, lr=0.1, tune=False, n_trials=20):
    import xgboost as xgb
    et = split.target_edge_type
    Xtr = _pair_features(data, et, split.train_label_index); ytr = split.train_label.cpu().numpy()
    Xva = _pair_features(data, et, split.val_label_index);   yva = split.val_label.cpu().numpy()
    Xte = _pair_features(data, et, split.test_label_index);  yte = split.test_label.cpu().numpy()
    base = dict(n_estimators=600, max_depth=max_depth, learning_rate=lr, subsample=0.9, colsample_bytree=0.9,
                eval_metric="logloss", tree_method="hist", random_state=seed, n_jobs=-1, early_stopping_rounds=30)
    if tune:
        import optuna
        def objective(trial):
            p = dict(n_estimators=600, max_depth=trial.suggest_int("max_depth",3,9),
                     learning_rate=trial.suggest_float("learning_rate",0.03,0.3,log=True),
                     subsample=trial.suggest_float("subsample",0.7,1.0),
                     colsample_bytree=trial.suggest_float("colsample_bytree",0.7,1.0),
                     min_child_weight=trial.suggest_int("min_child_weight",1,8),
                     eval_metric="logloss", tree_method="hist", random_state=seed, n_jobs=-1, early_stopping_rounds=30)
            m = xgb.XGBClassifier(**p); m.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
            return roc_auc_score(yva, m.predict_proba(Xva)[:,1])
        optuna.logging.set_verbosity(optuna.logging.WARNING)
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=False); base.update(study.best_params)
    model = xgb.XGBClassifier(**base); model.fit(Xtr, ytr, eval_set=[(Xva, yva)], verbose=False)
    return compute_all_metrics(yte, model.predict_proba(Xte)[:,1])
print("L8 ready")

### L9 · Multi-hop mechanism-path extractor

In [ ]:
from math import log2
PROT = "gene_protein"; PATHWAY = "pathway"
CARRIER_BLOCKLIST = frozenset({"ALB","A2M","AHSG","ORM1","ORM2","SERPINA1","GC","TTR","APOA1"})
DIRECT_TARGET_BLOCKLIST = CARRIER_BLOCKLIST
PROMISC_DRUG_DEG_CAP = 10**9

def _edge_pairs(data, et):
    ei = data[et].edge_index; return ei[0].tolist(), ei[1].tolist()

def build_mech_index(data):
    idx = {k: defaultdict(set) for k in ["drug2prot","dis2prot","prot2dis","ppi","prot2pw","pw2prot","prot2drug"]}
    ets = set(data.edge_types)
    et = (DRUG_TYPE,"drug_protein",PROT)
    if et in ets:
        s, d = _edge_pairs(data, et)
        for a, b in zip(s, d): idx["drug2prot"][a].add(b); idx["prot2drug"][b].add(a)
    et = (DISEASE_TYPE,"disease_protein",PROT)
    if et in ets:
        s, d = _edge_pairs(data, et)
        for a, b in zip(s, d): idx["dis2prot"][a].add(b); idx["prot2dis"][b].add(a)
    et = (PROT,"protein_protein",PROT)
    if et in ets:
        s, d = _edge_pairs(data, et)
        for a, b in zip(s, d): idx["ppi"][a].add(b); idx["ppi"][b].add(a)
    et = (PROT,"pathway_protein",PATHWAY)
    if et in ets:
        s, d = _edge_pairs(data, et)
        for a, b in zip(s, d): idx["prot2pw"][a].add(b); idx["pw2prot"][b].add(a)
    idx["prot_dis_deg"] = {p: len(v) for p, v in idx["prot2dis"].items()}
    idx["pw_size"] = {pw: len(v) for pw, v in idx["pw2prot"].items()}
    idx["prot_drug_deg"] = {p: len(v) for p, v in idx["prot2drug"].items()}
    return idx

def _mname(data, nt, i):
    names = getattr(data[nt], "node_names", None)
    return str(names[i]) if names is not None and i < len(names) else f"{nt}:{i}"

def _spec(deg): return 1.0/log2(deg+2)
def _drug_spec(dd): return 1.0/log2(dd+2)

def mechanism_paths(data, idx, drug_idx, disease_idx, max_paths=8, prot_dis_cap=150, pw_size_cap=300,
                    max_targets=40, promisc_drug_cap=PROMISC_DRUG_DEG_CAP):
    targets = list(idx["drug2prot"].get(drug_idx, set()))[:max_targets]
    dis_prots = idx["dis2prot"].get(disease_idx, set())
    drug_n = _mname(data, DRUG_TYPE, drug_idx); dis_n = _mname(data, DISEASE_TYPE, disease_idx)
    dd = idx.get("prot_drug_deg", {}); out = []
    def hub(p, pn): return (pn in CARRIER_BLOCKLIST or dd.get(p, 0) > promisc_drug_cap)
    for p in targets:
        if p in dis_prots:
            deg = idx["prot_dis_deg"].get(p, 1)
            if deg > prot_dis_cap: continue
            pn = _mname(data, PROT, p)
            if pn in DIRECT_TARGET_BLOCKLIST: continue
            out.append({"type":"direct_target","len":2,"score":3.0+_spec(deg)*_drug_spec(dd.get(p,0)),
                        "drug":drug_n,"disease":dis_n,"genes":[pn],"pathway":None,
                        "text":f"{drug_n} --targets--> {pn} <--associated-- {dis_n}"})
    for p1 in targets:
        p1n = _mname(data, PROT, p1)
        if hub(p1, p1n): continue
        for p2 in idx["ppi"].get(p1, set()) & dis_prots:
            deg = idx["prot_dis_deg"].get(p2, 1)
            if deg > prot_dis_cap: continue
            p2n = _mname(data, PROT, p2)
            if hub(p2, p2n): continue
            out.append({"type":"ppi","len":3,"score":2.0+_spec(deg)*_drug_spec(dd.get(p1,0)),
                        "drug":drug_n,"disease":dis_n,"genes":[p1n,p2n],"pathway":None,
                        "text":f"{drug_n} --targets--> {p1n} --interacts--> {p2n} <--associated-- {dis_n}"})
    for p1 in targets:
        p1n = _mname(data, PROT, p1)
        if hub(p1, p1n): continue
        for pw in idx["prot2pw"].get(p1, set()):
            pw_sz = idx["pw_size"].get(pw, 1)
            if pw_sz > pw_size_cap: continue
            members = {q for q in (idx["pw2prot"].get(pw, set()) & dis_prots) if not hub(q, _mname(data, PROT, q))}
            if not members: continue
            p2 = min(members, key=lambda q: idx["prot_dis_deg"].get(q, 1))
            pwn, p2n = _mname(data, PATHWAY, pw), _mname(data, PROT, p2)
            out.append({"type":"shared_pathway","len":4,"score":1.0+_spec(pw_sz)*_drug_spec(dd.get(p1,0)),
                        "drug":drug_n,"disease":dis_n,"genes":[p1n,p2n],"pathway":pwn,
                        "text":f"{drug_n} --targets--> {p1n} --in pathway--> {pwn} <--in pathway-- {p2n} <--associated-- {dis_n}"})
    out.sort(key=lambda d: -d["score"]); seen, uniq = set(), []
    for p in out:
        if p["text"] in seen: continue
        seen.add(p["text"]); uniq.append(p)
    return uniq[:max_paths]

def classify_support(paths):
    if any(p["type"]=="direct_target" for p in paths): return "direct-target mechanism"
    if any(p["type"]=="ppi" for p in paths): return "interaction-level mechanism"
    if any(p["type"]=="shared_pathway" for p in paths): return "pathway-level mechanism"
    return "no mechanistic path found"

def mechanism_score(paths): return max((p["score"] for p in paths), default=0.0)
print("L9 ready")

### L10 · Candidate ranking + 2-hop rationale extraction

In [ ]:
def _known_pairs(data):
    known = set()
    for et in data.edge_types:
        s, r, d = et
        if {s, d} == {DRUG_TYPE, DISEASE_TYPE} and r in _THERAPEUTIC_NORM:
            ei = data[et].edge_index
            if s == DRUG_TYPE:
                for a, b in zip(ei[0].tolist(), ei[1].tolist()): known.add((a, b))
            else:
                for a, b in zip(ei[0].tolist(), ei[1].tolist()): known.add((b, a))
    return known

@torch.no_grad()
def predict_candidates_for_diseases(model, data, target, disease_indices, device, top_k=10,
                                    exclude_known=True, rank_by="specificity", pop_sample=64, seed=0):
    model.eval(); z = model.encode(data); dev = z[DRUG_TYPE].device
    num_drugs = int(data[DRUG_TYPE].num_nodes); num_dis = int(data[DISEASE_TYPE].num_nodes)
    known = _known_pairs(data) if exclude_known else set(); all_drugs = torch.arange(num_drugs, device=dev)
    def score_disease(dz):
        eli = torch.stack([all_drugs, torch.full((num_drugs,), dz, device=dev)])
        return torch.sigmoid(model.decode(z, target, eli)).cpu()
    pop = None
    if rank_by == "specificity":
        g = torch.Generator().manual_seed(seed)
        sample = torch.randperm(num_dis, generator=g)[:min(pop_sample, num_dis)].tolist()
        acc = torch.zeros(num_drugs)
        for dz in sample: acc += score_disease(dz)
        pop = acc / max(1, len(sample))
    out = {}
    for dz in disease_indices:
        scores = score_disease(dz); rank_val = (scores - pop) if pop is not None else scores
        order = torch.argsort(rank_val, descending=True).tolist(); ranked = []
        for di in order:
            if exclude_known and (di, dz) in known: continue
            ranked.append((di, float(scores[di]), float(rank_val[di])))
            if len(ranked) >= top_k: break
        out[dz] = ranked
    return out

def _typed_neighbors(data, node_type, node_idx):
    nbrs = defaultdict(dict)
    for et in data.edge_types:
        s, r, d = et; ei = data[et].edge_index
        if s == node_type:
            for nb in ei[1][ei[0]==node_idx].tolist(): nbrs[d].setdefault(nb, r)
        if d == node_type:
            for nb in ei[0][ei[1]==node_idx].tolist(): nbrs[s].setdefault(nb, r)
    return nbrs

def extract_paths(data, drug_idx, disease_idx,
                  bridge_types=("gene_protein","pathway","biological_process","effect_phenotype"), max_paths=8):
    dn = _typed_neighbors(data, DRUG_TYPE, drug_idx); sn = _typed_neighbors(data, DISEASE_TYPE, disease_idx)
    paths = []
    for bt in bridge_types:
        for mid in set(dn.get(bt, {})) & set(sn.get(bt, {})):
            paths.append({"bridge_type":bt,"bridge_name":_mname(data,bt,mid),
                          "text":f"{_mname(data,DRUG_TYPE,drug_idx)} --{dn[bt][mid]}--> {_mname(data,bt,mid)} <--{sn[bt][mid]}-- {_mname(data,DISEASE_TYPE,disease_idx)}"})
    priority = {bt: i for i, bt in enumerate(bridge_types)}
    paths.sort(key=lambda p: priority.get(p["bridge_type"], 99))
    return paths[:max_paths]
print("L10 ready")

### L11 · UniProt→HGNC (mygene), DrugMechDB, mechanism-recovery supervision

In [ ]:
import json
_ACC_RE = re.compile(r"^(?:[OPQ][0-9][A-Z0-9]{3}[0-9]|[A-NR-Z][0-9](?:[A-Z][A-Z0-9]{2}[0-9]){1,2})$")
MYGENE_URL = "https://mygene.info/v3/query"; _BATCH = 800
_DMDB_URLS = ("https://raw.githubusercontent.com/SuLab/DrugMechDB/main/indication_paths.yaml",
              "https://raw.githubusercontent.com/SuLab/DrugMechDB/master/indication_paths.yaml")

def _uni_clean(acc):
    acc = str(acc).strip()
    if acc.lower().startswith("uniprot:"): acc = acc.split(":",1)[1]
    return acc.split("-",1)[0].strip().upper()

def _uni_load_cache():
    if UNIPROT_CACHE.exists():
        try:
            with open(UNIPROT_CACHE) as f: return json.load(f)
        except Exception: return {}
    return {}

def _uni_save_cache(c):
    UNIPROT_CACHE.parent.mkdir(parents=True, exist_ok=True)
    tmp = UNIPROT_CACHE.with_suffix(".json.tmp")
    with open(tmp, "w") as f: json.dump(c, f, indent=0, sort_keys=True)
    tmp.replace(UNIPROT_CACHE)

def uniprot_to_symbol(accessions, use_network=True):
    import requests
    cleaned = {a for a in (_uni_clean(a) for a in accessions) if a}
    cache = _uni_load_cache()
    missing = sorted(a for a in cleaned if a not in cache and _ACC_RE.match(a))
    if missing and use_network:
        resolved = {}
        try:
            for i in range(0, len(missing), _BATCH):
                chunk = missing[i:i+_BATCH]
                r = requests.post(MYGENE_URL, data={"q":",".join(chunk),"scopes":"uniprot",
                                  "fields":"symbol","species":"human"}, timeout=60)
                r.raise_for_status(); hits = r.json()
                if isinstance(hits, list):
                    for h in hits:
                        if isinstance(h, dict) and h.get("query") and h.get("symbol") and not h.get("notfound"):
                            q = str(h["query"]).upper()
                            if q not in resolved: resolved[q] = str(h["symbol"])
        except Exception: resolved = {}
        for acc in missing: cache[acc] = resolved.get(acc, "")
        _uni_save_cache(cache)
    return {a: cache[a] for a in cleaned if cache.get(a)}

def build_drugmechdb_drug_symbols():
    import requests, yaml
    raw = None
    for u in _DMDB_URLS:
        try:
            r = requests.get(u, timeout=60)
            if r.ok and len(r.text) > 1000: raw = r.text; break
        except Exception: continue
    if raw is None: return {}
    entries = yaml.safe_load(raw); accs, drug_accs = set(), {}
    for e in entries:
        drug = str(e.get("graph", {}).get("drug","")).strip().lower()
        if not drug: continue
        for n in e.get("nodes", []):
            nid = str(n.get("id",""))
            if nid.startswith("UniProt:"):
                a = nid.split(":",1)[1]; accs.add(a); drug_accs.setdefault(drug, set()).add(a)
    mp = uniprot_to_symbol(sorted(accs)); out = {}
    for drug, a_set in drug_accs.items():
        syms = {mp[a].upper() for a in a_set if mp.get(a)}
        if syms: out[drug] = syms
    return out

def symbol_to_gene_index(data):
    out = {}
    for i, nm in enumerate(data[GENE_TYPE].node_names):
        s = str(nm).strip().upper()
        if s and s not in out: out[s] = i
    return out

@dataclass
class MechExamples:
    drug: torch.Tensor; dis: torch.Tensor; gene: torch.Tensor; pairs: list

def build_mech_examples(data, pair_index, dmdb, sym2gidx):
    drug_names = [str(x).strip().lower() for x in data[DRUG_TYPE].node_names]
    drugs, diss, genes, pairs, seen = [], [], [], [], set()
    for c in range(pair_index.size(1)):
        di = int(pair_index[0,c]); ci = int(pair_index[1,c])
        if (di, ci) in seen: continue
        seen.add((di, ci))
        syms = dmdb.get(drug_names[di]) if di < len(drug_names) else None
        if not syms: continue
        gidx = sorted({sym2gidx[s] for s in syms if s in sym2gidx})
        if not gidx: continue
        pairs.append((di, ci, gidx))
        for g in gidx: drugs.append(di); diss.append(ci); genes.append(g)
    return MechExamples(torch.tensor(drugs, dtype=torch.long), torch.tensor(diss, dtype=torch.long),
                        torch.tensor(genes, dtype=torch.long), pairs)

class DegreeMatchedDecoys:
    def __init__(self, prot_drug_deg, num_genes, n_buckets=10, seed=0):
        self.rng = np.random.default_rng(seed)
        deg = np.array([prot_drug_deg.get(i,0) for i in range(num_genes)], dtype=float)
        order = np.argsort(deg, kind="stable"); bucket = np.empty(num_genes, dtype=int)
        edges = np.linspace(0, num_genes, n_buckets+1).astype(int)
        for b in range(n_buckets): bucket[order[edges[b]:edges[b+1]]] = b
        self.bucket = bucket; self.by_bucket = [np.where(bucket==b)[0] for b in range(n_buckets)]
    def sample(self, pos_gene, exclude, k):
        pool = self.by_bucket[self.bucket[pos_gene]]; out, tries = [], 0
        while len(out) < k and tries < k*50:
            c = int(pool[self.rng.integers(0, len(pool))])
            if c not in exclude and c not in out: out.append(c)
            tries += 1
        while len(out) < k:
            c = int(self.rng.integers(0, len(self.bucket)))
            if c not in exclude and c not in out: out.append(c)
        return out
print("L11 ready")

### L12 · Europe PMC retrieval, OA full text, LLM client, verifier

In [ ]:
import html
EUROPE_PMC = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
FULLTEXT_URL = "https://www.ebi.ac.uk/europepmc/webservices/rest/{source}/{id}/fullTextXML"
_TAG = re.compile(r"<[^>]+>"); _WS = re.compile(r"\s+"); _REFS = re.compile(r"<ref-list.*", re.DOTALL|re.IGNORECASE)
GRADES = ("supported","weak","contradicted","unknown")
_CUES = ("inhibit","inhibitor","target","targets","treatment","treat","therapy","therapeutic",
         "suppress","block","antagonist","agonist","mechanism","mutation","overexpress","activate")
_MOA_CUE = re.compile(r"\b(inhibit\w*|bind\w*|target\w*|substrate\w*|activat\w*|antagoni\w*|agoni\w*|block\w*|suppress\w*|degrad\w*|modulat\w*|abrogat\w*|disrupt\w*|phosphorylat\w*|stabili[sz]\w*|sequester\w*|interact\w*)\b", re.IGNORECASE)
_ASSOC_CUE = re.compile(r"\b(rs\d+|polymorphism\w*|genotyp\w*|snp|snps|variant\w*|allele\w*|gwas|haplotype\w*|prognos\w*|surviv\w*|associat\w*|susceptibilit\w*|predispos\w*|risk allele)\b", re.IGNORECASE)
_MECH_TERMS = ("mechanism OR inhibitor OR inhibits OR inhibition OR target OR targets OR agonist OR antagonist OR activates OR activation OR binds OR modulates")

def _clean_html(t): return html.unescape(_TAG.sub("", t or "")).strip()
def llm_available(): return bool(os.environ.get("ONCO_LLM_API_KEY"))

def llm_chat(messages, temperature=0.2, max_tokens=900, json_mode=False, use_cache=True):
    import requests
    key = os.environ.get("ONCO_LLM_API_KEY")
    if not key: return None
    base = os.environ.get("ONCO_LLM_BASE_URL", "https://api.openai.com/v1").rstrip("/")
    model = os.environ.get("ONCO_LLM_MODEL", "gpt-4o-mini")
    payload = {"model":model,"messages":messages,"temperature":temperature,"max_tokens":max_tokens}
    if json_mode: payload["response_format"] = {"type":"json_object"}
    LLM_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    ck = hashlib.md5(json.dumps({"u":base,"m":model,"p":payload}, sort_keys=True).encode()).hexdigest()
    cp = LLM_CACHE_DIR / f"{ck}.json"
    if use_cache and cp.exists(): return json.loads(cp.read_text())["content"]
    try:
        r = requests.post(f"{base}/chat/completions",
                          headers={"Authorization":f"Bearer {key}","Content-Type":"application/json"},
                          json=payload, timeout=90); r.raise_for_status()
        content = r.json()["choices"][0]["message"]["content"]
    except Exception as exc:
        print(f"  [llm] call failed: {exc}"); return None
    cp.write_text(json.dumps({"content":content})); return content

def search_literature(query, page_size=5, with_abstract=True):
    import requests
    try:
        r = requests.get(EUROPE_PMC, params={"query":query,"format":"json","pageSize":page_size,
                         "resultType":"core" if with_abstract else "lite"}, timeout=30)
        r.raise_for_status(); results = r.json().get("resultList", {}).get("result", [])
    except Exception as exc:
        print(f"  [lit] Europe PMC failed: {exc}"); return []
    return [{"title":_clean_html(it.get("title","")),"authors":it.get("authorString",""),
             "year":it.get("pubYear",""),"source":it.get("source",""),"id":it.get("id",""),
             "abstract":_clean_html(it.get("abstractText","")) if with_abstract else ""} for it in results]

def _drug_token(name): return re.split(r"\s*\(", name or "")[0].strip()
def _phrase(t): return '"' + t.replace('"',"").strip() + '"'
def _gene_in_text(sym, rec_or_text):
    if not sym: return False
    text = rec_or_text if isinstance(rec_or_text, str) else f"{rec_or_text.get('title','')} {rec_or_text.get('abstract','')}"
    return re.search(rf"\b{re.escape(sym)}\b", text, re.IGNORECASE) is not None

def _build_queries(drug, genes, disease):
    drug = _drug_token(drug); genes = [g for g in (genes or []) if g]; q = []
    if drug and genes:
        p = genes[0]
        q.append(f"{_phrase(drug)} AND {_phrase(p)}")
        q.append(f"{_phrase(drug)} AND {_phrase(p)} AND ({_MECH_TERMS})")
        q.append(f"{_phrase(drug)} AND (ABSTRACT:{_phrase(p)} OR TITLE:{_phrase(p)})")
        if len(genes) > 1: q.append(f"{_phrase(drug)} AND {_phrase(genes[1])}")
    elif drug: q.append(_phrase(drug))
    if drug and disease: q.append(f"{_phrase(drug)} AND {_phrase(_drug_token(disease))}")
    return q

def retrieve_for_mechanism(drug, genes, disease=None, n=6, per_query=6):
    queries = _build_queries(drug, genes, disease)
    if not queries: return []
    gene_syms = [g for g in (genes or []) if g]; seen, order = {}, []
    for q in queries:
        for rec in search_literature(q, page_size=per_query, with_abstract=True):
            rid = rec.get("id") or ""; key = rid or (rec.get("source","") + "|" + rec.get("title",""))
            if not key: continue
            if key not in seen: seen[key] = rec; order.append(key)
            elif rec.get("abstract") and not seen[key].get("abstract"): seen[key] = rec
    def tier(key):
        rec = seen[key]; has = bool(rec.get("abstract")); ng = any(_gene_in_text(g, rec) for g in gene_syms)
        return 0 if (has and ng) else (1 if has else 2)
    ranked = sorted(enumerate(order), key=lambda ik: (tier(ik[1]), ik[0]))
    return [seen[k] for _, k in ranked[:n]]

def _strip_xml(xml):
    return _WS.sub(" ", html.unescape(_TAG.sub(" ", _REFS.sub("", xml or "")))).strip()

def fetch_fulltext_record(source, rec_id, timeout=8.0):
    import requests
    if not source or not rec_id: return None
    try:
        r = requests.get(FULLTEXT_URL.format(source=source, id=rec_id), timeout=timeout)
        return _strip_xml(r.text) if (r.status_code == 200 and r.text) else None
    except Exception: return None

def fulltext_passages(records, gene_syms, max_papers=2, timeout=8.0, max_chars=6000):
    genes = [g for g in (gene_syms or []) if g]; out, tried = [], 0
    for rec in records:
        if tried >= max_papers: break
        source, rec_id = rec.get("source",""), rec.get("id","")
        if not source or not rec_id: continue
        tried += 1; text = fetch_fulltext_record(source, rec_id, timeout=timeout)
        if not text: continue
        passages = [p.strip() for p in re.split(r"(?<=[.!?])\s+", text) if any(_gene_in_text(g, p) for g in genes)]
        if not passages: continue
        out.append({"title":rec.get("title",""),"authors":rec.get("authors",""),"year":rec.get("year",""),
                    "source":source,"id":rec_id,"abstract":" ".join(passages)[:max_chars],"fulltext":True})
    return out

def _split_sentences(text): return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text or "") if s.strip()]

def mechanism_sentences(path, abstracts):
    drug = _drug_token(path.get("drug","")).lower(); genes = [g for g in path.get("genes",[]) if g]
    out, seen = [], set()
    for a in abstracts:
        text = f"{a.get('title','')}. {a.get('abstract','')}"
        for sent in _split_sentences(text):
            low = sent.lower()
            if not (drug and re.search(rf"\b{re.escape(drug)}\b", low)): continue
            hit = [g for g in genes if re.search(rf"\b{re.escape(g.lower())}\b", low)]
            if not hit or low in seen: continue
            seen.add(low)
            out.append({"sentence":sent,"source":a.get("source",""),"id":a.get("id",""),"genes":hit,
                        "mechanism_cue":bool(_MOA_CUE.search(sent)),
                        "association_only":bool(_ASSOC_CUE.search(sent)) and not _MOA_CUE.search(sent),
                        "fulltext":bool(a.get("fulltext"))})
    out.sort(key=lambda s: (not s["mechanism_cue"], s["association_only"])); return out

def lexical_grade(path, abstracts):
    corpus = " ".join(f"{a.get('title','')} {a.get('abstract','')}" for a in abstracts).lower()
    if not corpus.strip(): return {"grade":"unknown","evidence":"no literature retrieved"}
    genes = [g for g in path.get("genes",[]) if g]; drug = _drug_token(path.get("drug","")).lower()
    gene_hit = any(re.search(rf"\b{re.escape(g.lower())}\b", corpus) for g in genes)
    drug_hit = bool(drug) and drug in corpus; cue_hit = any(c in corpus for c in _CUES)
    hit_genes = [g for g in genes if re.search(rf"\b{re.escape(g.lower())}\b", corpus)]
    grade = ("supported" if gene_hit and drug_hit and cue_hit else
             "weak" if (gene_hit and (drug_hit or cue_hit)) or gene_hit or drug_hit else "unknown")
    return {"grade":grade,"evidence":f"co-mentions: drug={drug_hit}, genes={hit_genes or None}, mechanistic_cue={cue_hit}"}

def _llm_prompt(path, abstracts, sentences=None):
    lit = "\n\n".join(f"[{a['source']}:{a['id']}] {a['title']}\n{a['abstract'][:1200]}"
                      for a in abstracts if a.get("abstract")) or "(no abstracts retrieved)"
    focus = "\n".join(f"- [{s['source']}:{s['id']}] {s['sentence']}" for s in (sentences or [])) or "(no co-mention sentence)"
    sys = ("You are a strict biomedical evidence reviewer. Decide whether the text SUPPORTS the drug "
           "mechanism-of-action (MOA). Use ONLY the provided text. 'supported' REQUIRES an explicit MOA "
           "statement (drug inhibits/activates/binds/targets the protein). A pharmacogenetic/statistical "
           "association alone is 'weak'. 'weak' = co-occur, no MOA. 'contradicted' = drug does NOT act on "
           "target. 'unknown' = not addressed. Respond ONLY as JSON with keys grade "
           "(supported|weak|contradicted|unknown), evidence_quote (<=40-word verbatim), rationale (<=30 words).")
    usr = (f"Mechanism path:\n{path['text']}\n\nFocused sentences:\n{focus}\n\nFull abstracts:\n{lit}")
    return [{"role":"system","content":sys},{"role":"user","content":usr}]

def _parse_json(text):
    if not text: return {}
    try: return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        if m:
            try: return json.loads(m.group(0))
            except Exception: return {}
    return {}

def llm_grade(path, abstracts, sentences=None):
    if not llm_available(): return None
    res = _parse_json(llm_chat(_llm_prompt(path, abstracts, sentences), json_mode=True, temperature=0.0))
    if not res or res.get("grade") not in GRADES: return None
    return {"grade":res["grade"],"evidence":res.get("evidence_quote",""),"rationale":res.get("rationale","")}

def verify_mechanism(path, n_lit=5, use_llm=True):
    drug = _drug_token(path.get("drug","")); genes = path.get("genes",[]); disease = path.get("disease")
    abstracts = retrieve_for_mechanism(drug, genes, disease=disease, n=n_lit)
    ft = fulltext_passages(abstracts, genes, max_papers=2); pool = abstracts + ft
    sentences = mechanism_sentences(path, pool); lex = lexical_grade(path, abstracts)
    out = {"path":path["text"],"type":path.get("type"),
           "n_abstracts":len([a for a in abstracts if a.get("abstract")]),
           "lexical":lex,"llm":None,"grade":lex["grade"],"source":"lexical",
           "citations":[f"{a['source']}:{a['id']}" for a in abstracts if a.get("id")][:n_lit],
           "evidence_sentences":sentences,"fulltext_used":bool(ft)}
    if use_llm:
        g = llm_grade(path, abstracts, sentences)
        if g is not None: out["llm"] = g; out["grade"] = g["grade"]; out["source"] = "llm"
    return out
print("L12 ready — inlined library complete (no source-package dependency)")

---
# Workflow

Everything below uses only the inlined functions above.

## 2. PrimeKG: download, build, load, inspect

In [ ]:
download_primekg()
if not HETERODATA_PT.exists():
    build_hetero_from_primekg()
data, targets = load_primekg(with_features=True, force_fallback_features=USE_FALLBACK_FEATS)
target = targets["indication"]
print(graph_summary(data, targets))

In [ ]:
import pandas as pd
rows = [{"node_type":nt,"num_nodes":int(data[nt].num_nodes),
         "feature_dim":int(data[nt].x.size(1)) if "x" in data[nt] else None,
         "example":", ".join(str(n) for n in list(getattr(data[nt],"node_names",[]))[:2])}
        for nt in data.node_types]
display(pd.DataFrame(rows).sort_values("num_nodes", ascending=False).reset_index(drop=True))
print("oncology diseases:", int(data[DISEASE_TYPE].is_oncology.sum()),
      "| indication edges:", int(data[target].edge_index.size(1)))

## 3. Aim 1 — Candidate generation: link benchmark + ablations

We train the GNN, tuned XGBoost, MLP, and KGE across three regimes, aggregate over
`SEEDS`, run paired t-tests (GNN vs each baseline), and ablate the GNN's topology
and relation families. *Honest finding:* a tuned XGBoost on the same text features
matches/beats the GNN on link ranking.

In [ ]:
in_dims = {nt: int(data[nt].x.size(1)) for nt in data.node_types}
REGIMES = ["transductive", "inductive_cold_dst", "inductive_cold_src"]
REGIME_LABEL = {"transductive":"Transductive","inductive_cold_dst":"Inductive cold-disease (onc)","inductive_cold_src":"Inductive cold-drug"}
MODELS = ["GNN","XGBoost","MLP","KGE"]

def train_eval_one(name, split, seed):
    set_all_seeds(seed)
    if name == "GNN":
        m = HeteroGNN(list(data.node_types), list(split.base.edge_types), in_dims, hidden=128, num_layers=2, dropout=0.3)
        return evaluate_model(train_gnn(m, split, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE), split, DEVICE)
    if name == "MLP":
        m = FeatureMLP(list(data.node_types), in_dims, hidden=128, dropout=0.3)
        return evaluate_model(train_mlp(m, split, DEVICE, epochs=MLP_EPOCHS, patience=PATIENCE), split, DEVICE)
    if name == "KGE":
        m = DistMultKGE(DRUG_TYPE, DISEASE_TYPE, int(data[DRUG_TYPE].num_nodes), int(data[DISEASE_TYPE].num_nodes), dim=128)
        return evaluate_model(train_kge(m, split, DEVICE, epochs=KGE_EPOCHS, patience=PATIENCE), split, DEVICE)
    if name == "XGBoost":
        return run_xgboost(split, data, seed=seed, n_estimators=XGB_ESTIMATORS, tune=XGB_TUNE, n_trials=XGB_TRIALS)

per_seed = {r: {m: {} for m in MODELS} for r in REGIMES}
for regime in REGIMES:
    print(f"\n=== REGIME: {regime} ===")
    for seed in SEEDS:
        kw = {"restrict_oncology": True} if regime == "inductive_cold_dst" else {}
        split = make_split(data, target, regime, seed=seed, **kw)
        for name in MODELS:
            mt = train_eval_one(name, split, seed); per_seed[regime][name][seed] = mt
            print(f"  seed {seed} {name:8s} auroc={mt['auroc']:.4f} auprc={mt['auprc']:.4f}")

In [ ]:
import numpy as np
def agg(vals): a = np.asarray(vals, float); return a.mean(), a.std()
grid_mean = {}; print("Test AUROC (mean over seeds):")
table = {}
for regime in REGIMES:
    table[REGIME_LABEL[regime]] = {}
    for name in MODELS:
        vals = [per_seed[regime][name][s]["auroc"] for s in SEEDS]; m, sd = agg(vals)
        table[REGIME_LABEL[regime]][name] = m
import pandas as pd
df_grid = pd.DataFrame(table).T[MODELS].round(3); display(df_grid)

# Paired t-tests GNN vs baselines (AUROC), if >1 seed.
from scipy import stats as _stats
if len(SEEDS) > 1:
    print("\nPaired t-tests (GNN > baseline, AUROC):")
    for regime in REGIMES:
        gv = [per_seed[regime]["GNN"][s]["auroc"] for s in SEEDS]
        for b in ["XGBoost","MLP","KGE"]:
            bv = [per_seed[regime][b][s]["auroc"] for s in SEEDS]
            t, p = _stats.ttest_rel(gv, bv); p1 = p/2 if t > 0 else 1 - p/2
            print(f"  [{regime}] GNN vs {b}: mean_diff={np.mean(gv)-np.mean(bv):+.3f} p={p1:.4f}")

import matplotlib.pyplot as plt
ax = df_grid.plot(kind="bar", figsize=(10,5), rot=10); ax.axhline(0.5, color="gray", ls="--", lw=1)
ax.set_ylabel("Test AUROC"); ax.set_ylim(0.4, 1.0); ax.set_title("GNN vs baselines (PrimeKG link prediction)")
plt.tight_layout(); plt.savefig(FIGURES_DIR/"main_results.png", dpi=150, bbox_inches="tight"); plt.show()

In [ ]:
# Topology + relation ablations on the cold-disease regime (where topology should matter).
topo = {}
for mode in ["intact","shuffle","empty"]:
    vals = []
    for seed in ABLATION_SEEDS:
        sp = make_split(data, target, "inductive_cold_dst", seed=seed, restrict_oncology=True)
        s = sp if mode == "intact" else ablate_topology(sp, mode, seed=seed)
        set_all_seeds(seed)
        m = HeteroGNN(list(data.node_types), list(s.base.edge_types), in_dims, hidden=128, num_layers=2, dropout=0.3)
        vals.append(evaluate_model(train_gnn(m, s, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE), s, DEVICE)["auroc"])
    topo[mode] = float(np.mean(vals)); print(f"  topology[{mode:7s}] GNN AUROC={topo[mode]:.4f}")

rel_groups = {"drop_drug_protein":["drug_protein","carrier","enzyme","target","transporter"],
              "drop_disease_protein":["disease_protein"], "drop_pathway":["pathway"]}
rel = {}
for nm, subs in rel_groups.items():
    vals = []
    for seed in ABLATION_SEEDS:
        sp = make_split(data, target, "inductive_cold_dst", seed=seed, restrict_oncology=True)
        s = drop_relations(sp, subs); set_all_seeds(seed)
        m = HeteroGNN(list(data.node_types), list(s.base.edge_types), in_dims, hidden=128, num_layers=2, dropout=0.3)
        vals.append(evaluate_model(train_gnn(m, s, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE), s, DEVICE)["auroc"])
    rel[nm] = float(np.mean(vals)); print(f"  relation[{nm}] GNN AUROC={rel[nm]:.4f}")

plt.figure(figsize=(5.5,4)); plt.bar(list(topo), list(topo.values()), color=["#4C72B0","#DD8452","#C44E52"])
plt.axhline(0.5, color="gray", ls="--", lw=1); plt.ylabel("GNN AUROC (cold-disease)"); plt.ylim(0.4,1.0)
plt.title("Topology ablation"); plt.tight_layout(); plt.savefig(FIGURES_DIR/"ablation_topology.png", dpi=150, bbox_inches="tight"); plt.show()

## 4. Aim 2 — Mechanism extraction

In [ ]:
mech_idx = build_mech_index(data)
rxnames = list(data[DRUG_TYPE].node_names); dnames = list(data[DISEASE_TYPE].node_names)
ei = data[target].edge_index
dis2drugs = defaultdict(list)
for dr, ds in zip(ei[0].tolist(), ei[1].tolist()): dis2drugs[ds].append(dr)

QUERY = r"myeloid leukemia|promyelocytic|colorectal|small cell lung|breast carcinoma|glioblastoma"
shown = 0
for ds, drugs in dis2drugs.items():
    if not (dnames[ds] and re.search(QUERY, str(dnames[ds]), re.I)): continue
    paths = mechanism_paths(data, mech_idx, drugs[0], ds, max_paths=4)
    if not paths: continue
    print(f"\n[{rxnames[drugs[0]]} -> {dnames[ds]}]  ({classify_support(paths)}, score={mechanism_score(paths):.2f})")
    for p in paths: print(f"   ({p['type']}) {p['text']}")
    shown += 1
    if shown >= 5: break

## 5. Aim 4 — Mechanism separation + hard negatives

The falsifiable claim, LLM-free: true oncology indications carry stronger graph
mechanism structure than random pairs (published separation AUROC **0.879**).

In [ ]:
import random
rng = random.Random(0)
onco_set = set(torch.nonzero(data[DISEASE_TYPE].is_oncology).flatten().tolist())
known = _known_pairs(data)
true_pairs = [(dr, ds) for dr, ds in zip(ei[0].tolist(), ei[1].tolist()) if ds in onco_set]
rng.shuffle(true_pairs); true_pairs = true_pairs[:N_SEP]
num_drugs = int(data[DRUG_TYPE].num_nodes); onco_list = list(onco_set)
neg_pairs, seen = [], set()
while len(neg_pairs) < N_SEP and len(seen) < N_SEP*40:
    dr = rng.randrange(num_drugs); ds = rng.choice(onco_list)
    if (dr, ds) in known or (dr, ds) in seen: continue
    seen.add((dr, ds)); neg_pairs.append((dr, ds))

def score_group(pairs):
    s, direct = [], 0
    for dr, ds in pairs:
        paths = mechanism_paths(data, mech_idx, dr, ds, max_paths=6); s.append(mechanism_score(paths))
        if any(p["type"]=="direct_target" for p in paths): direct += 1
    return np.array(s), direct/max(1,len(pairs))

s_true, dt_true = score_group(true_pairs); s_neg, dt_neg = score_group(neg_pairs)
y = np.r_[np.ones_like(s_true), np.zeros_like(s_neg)]; auroc = roc_auc_score(y, np.r_[s_true, s_neg])
print(f"Separation AUROC (true vs random): {auroc:.3f}  [n={N_SEP}/group]")
print(f"  mean score    true={s_true.mean():.3f} random={s_neg.mean():.3f}")
print(f"  direct-target true={dt_true:.1%} random={dt_neg:.1%}")
print(f"  any-path      true={(s_true>0).mean():.1%} random={(s_neg>0).mean():.1%}")
import matplotlib.pyplot as plt
bins = np.linspace(0, max(s_true.max(),1e-3), 30); plt.figure(figsize=(7,4.2))
plt.hist(s_neg, bins=bins, alpha=0.6, label=f"random (mean {s_neg.mean():.2f})", color="#9aa0a6")
plt.hist(s_true, bins=bins, alpha=0.6, label=f"true (mean {s_true.mean():.2f})", color="#e8684a")
plt.xlabel("graph mechanism score"); plt.ylabel("count"); plt.legend()
plt.title(f"Mechanism separation (AUROC {auroc:.2f})")
plt.tight_layout(); plt.savefig(FIGURES_DIR/"mechanism_eval.png", dpi=150); plt.show()

In [ ]:
# DrugMechDB curated-mechanism agreement (UniProt->HGNC mapped). Needs network.
dmdb = build_drugmechdb_drug_symbols()
if dmdb:
    covered = agree = 0; examples = []
    for dr, ds in true_pairs:
        db = dmdb.get(str(rxnames[dr]).strip().lower())
        if not db: continue
        covered += 1
        ours = {g.upper() for p in mechanism_paths(data, mech_idx, dr, ds, max_paths=25) for g in p.get("genes", [])}
        ov = ours & db
        if ov:
            agree += 1
            if len(examples) < 3: examples.append((rxnames[dr], sorted(ov)))
    print(f"DrugMechDB agreement: {agree}/{covered} = {agree/max(1,covered):.3f}")
    for d, o in examples: print(f"   {d}: overlap={o}")
else:
    print("DrugMechDB unreachable; skipped.")

In [ ]:
# Hard negatives: degree-matched + oncology-drug, plus the no-direct-target ablation.
def node_degrees(node_type, num_nodes):
    deg = torch.zeros(num_nodes, dtype=torch.long)
    for et in data.edge_types:
        s, _, d = et; eix = data[et].edge_index
        if s == node_type: deg += torch.bincount(eix[0], minlength=num_nodes)
        if d == node_type: deg += torch.bincount(eix[1], minlength=num_nodes)
    return deg.numpy()

def decile_bins(degrees, members):
    members = list(members); vals = degrees[members]; edges = np.quantile(vals, np.linspace(0,1,11))[1:-1]
    bin_of, bucket = {}, {b: [] for b in range(10)}
    for n in members:
        b = min(int(np.digitize(degrees[n], edges, right=False)), 9); bin_of[n] = b; bucket[b].append(n)
    return bin_of, bucket

num_dis = int(data[DISEASE_TYPE].num_nodes)
drug_deg = node_degrees(DRUG_TYPE, num_drugs); dis_deg = node_degrees(DISEASE_TYPE, num_dis)
db_bin, db_bucket = decile_bins(drug_deg, range(num_drugs)); ds_bin, ds_bucket = decile_bins(dis_deg, onco_set)
onco_drugs = list({dr for dr, ds in zip(ei[0].tolist(), ei[1].tolist()) if ds in onco_set})

rnd = random.Random(2); dm = []
for dr_t, ds_t in true_pairs:
    dp = db_bucket.get(db_bin.get(dr_t), []); sp = ds_bucket.get(ds_bin.get(ds_t), [])
    if not dp or not sp: continue
    for _ in range(200):
        dr = rnd.choice(dp); ds = rnd.choice(sp)
        if (dr, ds) not in known: dm.append((dr, ds)); break
rnd = random.Random(3); od = []
while len(od) < N_SEP and len(od) < N_SEP:
    dr = rnd.choice(onco_drugs); ds = rnd.choice(onco_list)
    if (dr, ds) not in known: od.append((dr, ds))
    if len(od) >= N_SEP: break

def auroc_vs(neg):
    sg, _ = score_group(neg); yy = np.r_[np.ones_like(s_true), np.zeros_like(sg)]
    return roc_auc_score(yy, np.r_[s_true, sg])
print(f"random         AUROC={auroc:.3f}")
print(f"degree_matched AUROC={auroc_vs(dm):.3f}  (n={len(dm)})")
print(f"oncology_drug  AUROC={auroc_vs(od):.3f}  (n={len(od)})")
# no-direct-target ablation
def ndt_scores(pairs):
    return np.array([mechanism_score([p for p in mechanism_paths(data, mech_idx, dr, ds, max_paths=25) if p["type"]!="direct_target"]) for dr, ds in pairs])
ndt_t, ndt_n = ndt_scores(true_pairs), ndt_scores(neg_pairs)
print(f"no_direct_target (vs random) AUROC={roc_auc_score(np.r_[np.ones_like(ndt_t),np.zeros_like(ndt_n)], np.r_[ndt_t,ndt_n]):.3f}")

## 6. Aim 3 — LLM / lexical evidence verification

Verify one true mechanism path against Europe PMC abstracts. Works without a key
(lexical grounding); set `RUN_LLM=True` + a key for the strict LLM reviewer.

In [ ]:
chosen = None
for ds, drugs in dis2drugs.items():
    if not (dnames[ds] and re.search(QUERY, str(dnames[ds]), re.I)): continue
    paths = mechanism_paths(data, mech_idx, drugs[0], ds, max_paths=3)
    if paths and paths[0]["type"] == "direct_target": chosen = paths[0]; break
print("Verifying:", chosen["text"], "\n")
v = verify_mechanism(chosen, n_lit=4, use_llm=(RUN_LLM and bool(os.environ.get("ONCO_LLM_API_KEY"))))
print("grade        :", v["grade"], "(source:", v["source"] + ")")
print("abstracts    :", v["n_abstracts"], "| citations:", v["citations"][:3])
print("lexical      :", v["lexical"]["grade"], "->", v["lexical"]["evidence"])
if v.get("llm"): print("LLM          :", v["llm"]["grade"], "->", v["llm"].get("evidence","")[:160])
print("\nTop co-mention sentences:")
for s in v["evidence_sentences"][:3]:
    print(f"   [{s['source']}] (moa_cue={s['mechanism_cue']}) {s['sentence'][:150]}")

## 7. Finding 4 — Blinded mechanism recovery (the graph's genuine edge)

Train the GNN jointly (link + a contrastive loss ranking the curated DrugMechDB
bridge gene above degree-matched decoys), then name the bridge gene for held-out
pairs — unblinded and **mechanism-blinded** (direct drug→target edge removed). A
tabular model has no analogue. Uses real (ST) features.

In [ ]:
# Mechanism recovery needs gene embeddings -> use REAL ST features.
data_st, targets_st = load_primekg(with_features=True, force_fallback_features=False)
target_st = targets_st["indication"]; in_dims_st = {nt: int(data_st[nt].x.size(1)) for nt in data_st.node_types}
mech_idx_st = build_mech_index(data_st); num_genes = int(data_st[GENE_TYPE].num_nodes)
dmdb = dmdb if dmdb else build_drugmechdb_drug_symbols()
sym2gidx = symbol_to_gene_index(data_st)
print(f"DrugMechDB drugs mapped: {len(dmdb)} | genes: {num_genes}")
gnames = list(data_st[GENE_TYPE].node_names); rxnames_st = list(data_st[DRUG_TYPE].node_names); dnames_st = list(data_st[DISEASE_TYPE].node_names)
DP = (DRUG_TYPE, "drug_protein", GENE_TYPE); PD = (GENE_TYPE, "drug_protein", DRUG_TYPE)
KS = [5, 10, 20]

def positives(eli, lab): return eli[:, lab.bool()]

def rank_metrics(scores, true_genes):
    order = torch.argsort(scores, descending=True).tolist(); pos = {g: i for i, g in enumerate(order)}
    best = min((pos[g] for g in true_genes if g in pos), default=None)
    rec = {k: (1.0 if any(pos.get(g, 1e9) < k for g in true_genes) else 0.0) for k in KS}
    return rec, (1.0/(best+1) if best is not None else 0.0)

@torch.no_grad()
def gnn_scores(model, z, di, ci):
    g = torch.arange(num_genes); d = torch.full((num_genes,), di); c = torch.full((num_genes,), ci)
    return model.score_mechanism(z, d, g, c).detach().cpu()

def blind_base(base, remove):
    nb = HeteroData()
    for nt in base.node_types:
        for k, v in base[nt].items(): nb[nt][k] = v
    for et in base.edge_types:
        eix = base[et].edge_index
        if et == DP:
            keep = [j for j in range(eix.size(1)) if (int(eix[0,j]), int(eix[1,j])) not in remove]
            nb[et].edge_index = eix[:, torch.tensor(keep, dtype=torch.long)] if keep else eix[:, :0]
        elif et == PD:
            keep = [j for j in range(eix.size(1)) if (int(eix[1,j]), int(eix[0,j])) not in remove]
            nb[et].edge_index = eix[:, torch.tensor(keep, dtype=torch.long)] if keep else eix[:, :0]
        else: nb[et].edge_index = eix
    return nb

def drug2prot_from(base):
    d2p = defaultdict(set)
    if DP in base.edge_types:
        eix = base[DP].edge_index
        for a, b in zip(eix[0].tolist(), eix[1].tolist()): d2p[a].add(b)
    return d2p
print("recovery helpers ready")

In [ ]:
def eval_systems(joint, linkonly, base, test_pairs):
    zj = joint.encode(base); zl = linkonly.encode(base); d2p = drug2prot_from(base)
    agg = {s: {"rec": {k: [] for k in KS}, "mrr": []} for s in ["joint_gnn","linkonly_gnn","target_lookup","degree_prior"]}
    deg = torch.tensor([mech_idx_st["prot_drug_deg"].get(i, 0) for i in range(num_genes)], dtype=float)
    for di, ci, genes in test_pairs:
        tg = set(genes)
        for name, sc in [("joint_gnn", gnn_scores(joint, zj, di, ci)),
                         ("linkonly_gnn", gnn_scores(linkonly, zl, di, ci))]:
            r, m = rank_metrics(sc, tg)
            for k in KS: agg[name]["rec"][k].append(r[k])
            agg[name]["mrr"].append(m)
        tgt = d2p.get(di, set()); s = torch.rand(num_genes)*0.5
        if tgt: s[torch.tensor(sorted(tgt), dtype=torch.long)] += 1.0
        r, m = rank_metrics(s, tg)
        for k in KS: agg["target_lookup"]["rec"][k].append(r[k])
        agg["target_lookup"]["mrr"].append(m)
        r, m = rank_metrics(deg + torch.rand(num_genes)*1e-3, tg)
        for k in KS: agg["degree_prior"]["rec"][k].append(r[k])
        agg["degree_prior"]["mrr"].append(m)
    return {s: {"recall": {k: float(np.mean(agg[s]["rec"][k])) for k in KS}, "mrr": float(np.mean(agg[s]["mrr"]))} for s in agg}

seed_results = []
for seed in REC_SEEDS:
    set_all_seeds(seed)
    sp = make_split(data_st, target_st, "inductive_cold_dst", seed=seed, restrict_oncology=True)
    mech_tr = build_mech_examples(data_st, positives(sp.train_label_index, sp.train_label), dmdb, sym2gidx)
    mech_te = build_mech_examples(data_st, positives(sp.test_label_index, sp.test_label), dmdb, sym2gidx)
    if not mech_te.pairs:
        print(f"seed {seed}: no covered held-out pairs"); continue
    decoys = DegreeMatchedDecoys(mech_idx_st["prot_drug_deg"], num_genes, seed=seed)
    joint = HeteroGNN(list(data_st.node_types), list(sp.base.edge_types), in_dims_st).to(DEVICE)
    joint = train_gnn_joint(joint, sp, DEVICE, mech_tr, decoys, lam=1.0, epochs=REC_EPOCHS)
    linkonly = HeteroGNN(list(data_st.node_types), list(sp.base.edge_types), in_dims_st).to(DEVICE)
    linkonly = train_gnn(linkonly, sp, DEVICE, epochs=REC_EPOCHS)
    unb = eval_systems(joint, linkonly, sp.base, mech_te.pairs)
    remove = {(di, g) for (di, ci, gs) in mech_te.pairs for g in gs}
    bli = eval_systems(joint, linkonly, blind_base(sp.base, remove), mech_te.pairs)
    seed_results.append((unb, bli))
    print(f"seed {seed}: test_pairs={len(mech_te.pairs)}")
    print(f"  UNBLINDED R@10  joint={unb['joint_gnn']['recall'][10]:.3f} linkonly={unb['linkonly_gnn']['recall'][10]:.3f} target_lookup={unb['target_lookup']['recall'][10]:.3f}")
    print(f"  BLINDED   R@10  joint={bli['joint_gnn']['recall'][10]:.3f} linkonly={bli['linkonly_gnn']['recall'][10]:.3f} target_lookup={bli['target_lookup']['recall'][10]:.3f}")

if seed_results:
    jb = np.mean([b["joint_gnn"]["recall"][10] for _, b in seed_results])
    print(f"\nBlinded joint-GNN R@10 (mean over {len(seed_results)} seed[s]): {jb:.3f} "
          f"-- where trivial/link-only baselines collapse to ~0.")

## 8. Deliverable — vetted oncology repurposing shortlist

In [ ]:
set_all_seeds(0)
dep_split = make_split(data, target, "transductive", seed=0, val_frac=0.1, test_frac=0.0)
dep = HeteroGNN(list(data.node_types), list(dep_split.base.edge_types), in_dims, hidden=128, num_layers=2, dropout=0.3)
dep = train_gnn(dep, dep_split, DEVICE, epochs=GNN_EPOCHS, patience=PATIENCE)

def pick_disease(substr):
    deg = defaultdict(int)
    for d in ei[1].tolist(): deg[d] += 1
    cands = [i for i, n in enumerate(dnames) if n and substr.lower() in str(n).lower() and i in onco_set]
    return max(cands, key=lambda i: deg[i]) if cands else None

disease_idx = [d for d in (pick_disease("glioblastoma"), pick_disease("pancreatic")) if d is not None]
print("Diseases:", [dnames[d] for d in disease_idx])
preds = predict_candidates_for_diseases(dep, data, target, disease_idx, DEVICE, top_k=5, exclude_known=True)

use_llm = RUN_LLM and llm_available()
rows = []
for dz in disease_idx:
    for drug_i, score, lift in preds[dz]:
        paths = extract_paths(data, drug_i, dz, max_paths=2)
        row = {"disease":dnames[dz],"candidate_drug":rxnames[drug_i],"model_score":round(score,3),
               "specificity_lift":round(lift,3),
               "top_KG_rationale":paths[0]["text"] if paths else "(no 2-hop bridge found)"}
        if use_llm:
            rep = build_candidate_report(rxnames[drug_i], dnames[dz], score, paths, use_llm=True)
            j = rep.get("judge") or {}; row["judge_plausibility"] = j.get("plausibility")
        rows.append(row)
import pandas as pd
display(pd.DataFrame(rows))

## Wrap-up

This notebook ran the **entire OncoEvidence pipeline with no dependency on the
project source package** — every model, split, metric, mechanism extractor, and
verifier was defined inline above. Artifacts are written to `data/`, `models/`,
`results/`, and `figures/`.

For the **published numbers**, set `FAST_MODE = False` (5 seeds, real ST features,
XGBoost tuning) and, with a key, `RUN_LLM = True`.

**Findings:** (1) a tuned XGBoost matches the GNN on link ranking; (2) only the
graph yields a traceable mechanism; (3) mechanism structure separates true vs
random oncology pairs (AUROC 0.879); (4) blinded mechanism recovery is a genuine
graph-only edge.

> *All predictions are hypothesis-generating and not medical advice.*